# 22: Iterators and some of their uses

Author: Paul Magwene and Greg Wray  
Date: 2026-MAR-13 


## Iterators   
   
Python programming uses **iterators** extensively. Iterators are objects that:
- represent streams of data (i.e., they do not hold data in memory)
- return the data they hold one element at a time when called by `next()`      
- keep track of which elements have already been returned                                                      
- raise a `StopIteration` exception if there is a call by `next()` after all elements have been returned 

Iterators are useful in many situations. For example, Python uses iterators behind the scenes to implement for loops and list comprehensions. Understanding how iterators work will help you write, read, and debug code. Because iterators load only a single element into memory at a time, they are particularly useful for very large data structures and for infinite sequences. 

### Constructing iterators from data objects

Standard Python data structures are *not* iterators themselves, but are called **iterable** because they can be used to *construct* iterator objects. The simplest way to think about an iterable is that it is something you can place on the right hand side of a for loop statement.

Iterable data objects include strings, unordered containers (set, frozenset), and ordered containers (range, list, tuple, dictionary). When you use one of these iterable data structures in a for loop or a list comprehension, Python constructs an interator object behind the scenes. 

Technically, an iterable is any Python object where you can call the `iter()` function to create an iterator. 

In [1]:
# lists are iterable
l = [1, 2, 3]                   
l_iter = iter(l)               
l_iter

In [2]:
# tuples are iterable
import math
t = (math.sin, math.cos, math.tan)   
t

(<function math.sin(x, /)>,
 <function math.cos(x, /)>,
 <function math.tan(x, /)>)

In [3]:
t_iter = iter(t)
t_iter

In [4]:
# strings are iterable
s = "ATGCAATGC"
s_iter = iter(s)        
s_iter

In [5]:
# dictionaries are iterable
d = {'a': 1, 'b': 2, 'c': 3}
d_iter = iter(d)        
d_iter

In [6]:
# sets are iterable
st = {-4, 0, 3, 5, 1, 2, 8, -2, -3}
st_iter = iter(st)      
st_iter                   

In [27]:
# integers and many other objects are not iterable
i_iter = iter(10)

TypeError: 'int' object is not iterable

### Retrieving elements from an iterator

The `next()` function returns items from an iterator. 

Iterators constructed from *ordered* objects return elements in the expected sequence. Iterators constructed from *unordered* objects like sets return elements in random sequence; nonetheless, the iterator still keeps track of which elements have been returned and will raise a `StopIteration` exception once all elements have been exhausted.

In [130]:
l = [1, 2, 3]
l_iter = iter(l)

In [131]:
list(l_iter)

[1, 2, 3]

In [126]:
next(l_iter)

1

In [127]:
next(l_iter)

2

In [128]:
next(l_iter)

3

In [129]:
next(l_iter)

StopIteration: 

In [7]:
# unordered objects like sets return items in random order
st = {-4, 0}
st_iter = iter(st)

In [8]:
next(st_iter)

0

In [9]:
next(st_iter)

-4

In [10]:
next(st_iter)

StopIteration: 

### for loops and comprehensions can be applied to any iterable

You can think if for loops and comprehensions as repeatedly calling `next()` on the iterator until a `StopIteration` exception is raised. Behind the scenes, Python uses the equivalent of a `try...except` structure to ensure that program execution is not interrupted.

In [70]:
# standard for loop
s = "ABCD"
for i in s:
    print(i)

A
B
C
D


In [11]:
# above is equivalent to
s = "ABCD"
s_iter = iter(s)
while True:                  # a potentially infinite loop!
    try:
        print(next(s_iter))
    except StopIteration:    # the loop is ended after the last item has been returned
        break

A
B
C
D


## Generators

In Python a **generator** is a special function that returns an iterator. Similar to typical iterators, generators only return results when asked. Unlike typical iterators, which passively return values, a generator function carries out *computation* on items in a data object before returning them. Generators are useful when you want to carry out computationally intensive operations on each element in a very large data object or a potentially infinite series of elements. 

In [26]:
# to use generators, we need the sys module
import sys

### Create a generator using a function

There are two ways to create a generator, each useful in certain circumstances. The most flexible way is to define a function and include a `yeild` statement instead of `return` statement. This approach allows you to define a multi-line function.   

Once the function has been defined, you can create a generator object by assigning a function call to a variable name. This object is a generator. You can create as many individual generators as you need using the same function.

In [76]:
# define a generator function
def my_gen(n):
    for i in range(n):
        val = math.cos(math.sin(i**2))      # stand-in for computationally intensive task
        yield val                           # return the first result and then pause

# instantiate a generator for values 0 through 9
x = my_gen(10)

# instantiate a generator for values 0 through 4
x = my_gen(5)
type(x)

generator

Similar to iterators, we can retrieve the next result from a generator by passing it to the `next()` function. 

When the function is called, the `yield` statement carries out the computation specified within the function on the first value and returns the result. The next time the function is called, the computation is applied to the second value, and so forth, until all the values have been processed.  


In [77]:
# now we can ask for individual returns
print(next(x))
print(next(x))
print(next(x))
print(next(x))
print(next(x))
print(next(x))

1.0
0.6663667453928805
0.7270351311688124
0.9162743174606308
0.9588413200803038


StopIteration: 

In [78]:
# alternatively, retrieve returns using a for loop; this automatically ends without an exception 
for k in my_gen(5):
    print(k)

1.0
0.6663667453928805
0.7270351311688124
0.9162743174606308
0.9588413200803038


### Create a generator using a one-line statement

The second way to create a generator is a one-line statement. The syntax is similar to a list comprehension, but uses parentheses (round brackets) instead of square brackets. Instead of returning a list, this returns a generator object. 

In [25]:
# first, define a conventional function
def complex_func(i):
    return math.cos(math.sin(i**2))      # stand-in for computationally intensive task

In [26]:
# create the generator object using round brackets / parentheses
my_gen = (complex_func(i) for i in range(100))
my_gen

<generator object <genexpr> at 0x131c297d0>

Now we can retrieve items one at a time using `next()`.

In [27]:
# note that Python automatically bundles multiple returns into a tuple
next(my_gen), next(my_gen), next(my_gen)

(1.0, 0.6663667453928805, 0.7270351311688124)

### Generators can save memory and time

Because generators only provide computed results "on demand", they are most useful for tasks that are memory intensive. A common use case is generating many trials of a simulation and extracting some information from each trial. The following example illustrates this using random arrays as a stand-in for repeated simulations. 

Although generators can save enormous amounts of memory, they are not suitable for all situations. If you need repeated access to any element or the ability to access items out of order, you will need to use a standard data structure like a list or tuple rather than a generator. 

In [28]:
import numpy as np
import random, sys

# normal function to construct a large array of random numbers
def make_big_array():
    return np.random.rand(100,100) 

# generator function to construct a large array of random numbers
def generate_big_array():
    yield np.random.rand(100,100) 

In [33]:
# make a list of large arrays
my_list = []
for i in range(10_000):
    my_list.append(make_big_array())

# how much memory (in bytes) does it occupy?
sys.getsizeof(my_list)

85176

In [34]:
# instantiate a generator
my_gen = (generate_big_array() for i in range(10_000))

# how much memory does it occupy?
sys.getsizeof(my_gen)

200

In [35]:
# retrieve the first 3 arrays
my_list[0:3]

[array([[8.41536960e-04, 6.26157663e-01, 3.78744009e-01, ...,
         1.95505349e-01, 4.69447537e-01, 8.76891107e-01],
        [3.38136286e-01, 3.49441961e-01, 5.13216604e-01, ...,
         3.86157689e-01, 7.21579123e-01, 6.51938819e-01],
        [2.69865466e-02, 3.16085292e-01, 7.48414553e-01, ...,
         2.01497374e-01, 6.95893342e-01, 8.32182113e-01],
        ...,
        [4.50620184e-01, 9.60595058e-01, 1.76138823e-01, ...,
         5.53414030e-01, 7.23448591e-01, 1.89577308e-01],
        [7.27301840e-02, 3.47772535e-01, 6.48950412e-01, ...,
         3.44430643e-02, 4.96651265e-01, 4.97256366e-01],
        [8.22979953e-01, 5.67340368e-02, 2.78814190e-01, ...,
         1.91475954e-01, 2.26148558e-01, 3.78055788e-01]]),
 array([[0.37519324, 0.91616189, 0.88211177, ..., 0.52369374, 0.45010933,
         0.51813473],
        [0.8799613 , 0.33248398, 0.19961132, ..., 0.57557577, 0.50368756,
         0.39247685],
        [0.95451628, 0.69811265, 0.15323225, ..., 0.04974577, 0.70802257,

In [38]:
# retrieve the first 3 arrays
list(next(my_gen))

[array([[0.18446904, 0.27471432, 0.7127988 , ..., 0.00138757, 0.69362772,
         0.87946482],
        [0.58921322, 0.06818015, 0.82485684, ..., 0.70219882, 0.86271108,
         0.98586123],
        [0.60361535, 0.06580041, 0.23730853, ..., 0.22274244, 0.48584662,
         0.88271893],
        ...,
        [0.71192124, 0.60740197, 0.88312232, ..., 0.32005199, 0.01854161,
         0.56438124],
        [0.42938347, 0.45891599, 0.67107507, ..., 0.80741566, 0.77190746,
         0.19044462],
        [0.94088415, 0.30274605, 0.70492444, ..., 0.88353757, 0.00727165,
         0.06245379]])]

## The `itertools` module

The `itertools` module is part of the Python standard library (see documentation [here](https://docs.python.org/3/library/itertools.html)). It includes a large number of functions that produce various useful iterators. We'll illustrate a few of these.

In [40]:
import itertools

### Infinite iterators

In [33]:
# This iterator infinitely returns the same thing!

rptr =  itertools.repeat("Hello")

[next(rptr) for i in range(5)]


['Hello', 'Hello', 'Hello', 'Hello', 'Hello']

In [54]:
# An example where a repeater can be useful in combination
# with a generator expression

import random

rep1 = itertools.repeat(1)
random_nuc = (random.choice("ATGC") for i in rep1)  # create a generator

# an infinite stream of random nucleotides!
next(random_nuc), next(random_nuc), next(random_nuc)

('C', 'A', 'T')

In [64]:
# using string join method with the generator expression above
# to generate a random 50bp nuc acid sequence 
# this will be different every time it's evaluated

rand_seq = ''.join(next(random_nuc) for i in range(50))
rand_seq

'ATGACAACACGTGTTGCTCCTTCTTAGTCCTCTGTTGCCCCATACTTACC'

In [5]:
z = itertools.count(11, step=3)

In [68]:
# itertools.cycle is another infinite iterator
# but one which cycles through its inputs in a defined order

color_cycle = itertools.cycle(["red", "green", "blue"])

next(color_cycle), next(color_cycle), next(color_cycle), next(color_cycle)

('red', 'green', 'blue', 'red')

In [70]:
# itertools.count gets a starting value
# and a step size and can inifitely return
# the next value in a sieres

ctr = itertools.count(0, step=9)
[next(ctr) for i in range(10)]


[0, 9, 18, 27, 36, 45, 54, 63, 72, 81]

In [71]:
# remembers where it was if called again
[next(ctr) for i in range(10)]

[90, 99, 108, 117, 126, 135, 144, 153, 162, 171]

### Other useful iterators 

In [52]:
# itertools.batched takes items in batches of size n from 
# the input iterable

seq = "ATGCATTTGACTC"

codon_itr = itertools.batched(seq, 3)

next(codon_itr), next(codon_itr), next(codon_itr)


(('A', 'T', 'G'), ('C', 'A', 'T'), ('T', 'T', 'G'))

In [74]:
# itertools.groupby provides an iterator
# that groups elements by a key function
#
# NOTE: items should be sorted by the same key function first

# first example, group by first letter

animals = ["aardvark", "ant", "dog", 
           "cat", "cougar", "koala", "beaver", "bear"]

sorted_by_first = sorted(animals, key = lambda x: x[0])

grp_by_first = itertools.groupby(sorted_by_first, key = lambda x: x[0])

for first, grp in grp_by_first:
    print("First letter:", first, " -> Group: ", list(grp))

First letter: a  -> Group:  ['aardvark', 'ant']
First letter: b  -> Group:  ['beaver', 'bear']
First letter: c  -> Group:  ['cat', 'cougar']
First letter: d  -> Group:  ['dog']
First letter: k  -> Group:  ['koala']


In [76]:
# second example, group by length of name

sorted_by_len = sorted(animals, key = len)

grp_by_len = itertools.groupby(sorted_by_len, key = len)

for namelen, grp in grp_by_len:
    print("Name length:", namelen, " -> Group: ", list(grp))


Name length: 3  -> Group:  ['ant', 'dog', 'cat']
Name length: 4  -> Group:  ['bear']
Name length: 5  -> Group:  ['koala']
Name length: 6  -> Group:  ['cougar', 'beaver']
Name length: 8  -> Group:  ['aardvark']


### Products, Permutations, and Combinations in itertools

In [31]:
# product (permutation with repetition) -> 
#    gove all possible sequences of length 2, 
#    composed of the characters drawn from "ABC"

list(itertools.product("ABC", repeat=2))

[('A', 'A'),
 ('A', 'B'),
 ('A', 'C'),
 ('B', 'A'),
 ('B', 'B'),
 ('B', 'C'),
 ('C', 'A'),
 ('C', 'B'),
 ('C', 'C')]

In [32]:
# permutation (w/out repetition) -> 
#    give  all possible sequences of length 2, 
#    composed of the characters drawn from "ABC"
#    but with no character appearing more than once

list(itertools.permutations("ABC", 2))

[('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]

In [34]:
# combination  -> 
#    give  all possible sequences of length 2, 
#    composed of the characters drawn from "ABC.
#    Order doesn't matter and no character appears
#    more than once.

list(itertools.combinations("ABC", 2))

[('A', 'B'), ('A', 'C'), ('B', 'C')]

In [36]:
# combination_with_replacement  -> 
#    give all possible sequences of length 2, 
#    composed of the characters drawn from "ABC.
#    Order doesn't matter and but character appears
#    can be repeated

list(itertools.combinations_with_replacement("ABC", 2))

[('A', 'A'), ('A', 'B'), ('A', 'C'), ('B', 'B'), ('B', 'C'), ('C', 'C')]

## The `pathlib` module

The `pathlib` module is another useful Python standard library (see documentation [here](https://docs.python.org/3/library/pathlib.html)). It provides a convenient object-oriented interface for working with file system paths. Representing as data objects rather than static strings provides some useful properties:

* paths and relative paths can be retrieved automatically
* paths can be manipulated in useful ways
* paths can be system-agnostic
* an iterator-based interface provides path information and the ability to search for matches

`pathlib` interacts well with the `os` module, which provides operations based on paths, such as reading and writing to files in other directories and navigating the file system. Functions and methods in the `os` module accept paths as either strings or path objects generated by `pathlib`. Path objects also work with Python's builtin functions and control flow structures. For instance, you can pass a path object directly to the context managers we have been using to open files for reading and writing.

In [2]:
from pathlib import Path
import os

### Creating path objects

Use the `Path()` creator function to specify a path from scratch. Pass a string that contains all or any part of path. A file name can optionally be included at the end of a path. 

In [11]:
# create a path object
p = Path("/Applications")
p

PosixPath('/Applications')

In [12]:
# create a path object to my home directory
p = Path.home()
p

PosixPath('/Users/gwray')

In [13]:
# create a path the current working directory
p = Path.cwd()
p

PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/22_Py_control&funs')

A path object is not simply a string. For example, it is aware of the current operating system. Asking for the type of path object will return a different result depending on whether you are running Windows or a Unix/Linux operating system (indicated by `Posix`.) 

In [56]:
# p is a path object
type(p)

pathlib.PosixPath

Note that `Path()` does *not* check whether a path actually exists before it creates a path object. This provides a lot of flexibility; for example, you can create a path object and then use it to create a new folder, or you can design a path object to work on a different file system from the one you are using.

In [57]:
# test whether a path object corresponds to a real path in the current file system
p.exists()

True

### Manipulating path objects

Once created, a path object can be altered or used to create a different path. The examples below illustrate a few of the basic operations; consult the documentation for many more.

In [61]:
# use the / operator to append a subdirectory to an existing path
proj_path = Path.home() / "projects" / "thesis_chapter_1"
proj_path

PosixPath('/Users/gwray/projects/thesis_chapter_1')

In [62]:
# use the .parent attribute to retrieve the parent (enclosing) directory
proj_path.parent  

PosixPath('/Users/gwray/projects')

In [63]:
# or the grandparent directory
proj_path.parent.parent

PosixPath('/Users/gwray')

A very useful feature is the ability to retrieve the absolute (complete) path of any item in the file structure.

In [7]:
# create a path object for a file in the current directory
short_path = Path("./protfile.csv")

In [6]:
# retrieve the absolute path
short_path.absolute()

PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/22_Py_control&funs/protfile.csv')

### Iterating over path objects

`pathlib` can also create iterator objects that return items in a directory. This is useful for learning directory contents and for filtering or modifying them. 

In [8]:
# create a path object to an example directory
# pick a directory that contains a mix of files and directories 
bio724 = Path.cwd().parent.parent
bio724

PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25')

Use the `.iterdir()` method to create a directory iterator object. You can then filter the results based on file type.

In [9]:
# use a list comprehension with a directory iterator 
directories = [i for i in bio724.iterdir()]
directories

[PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/.DS_Store'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/NotesForNextIteration.pages'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Notes.pages'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Fall_project_desc.key'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Lessons'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/z_fall_proposals'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Future'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Data_lunches'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Planner.numbers')]

In [10]:
# use a list comprehension with a directory iterator 
directories = [i for i in bio724.iterdir() if i.is_dir()]
directories

[PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Lessons'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/z_fall_proposals'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Future'),
 PosixPath('/Users/gwray/Documents/Courses/Bio_724D_25/Data_lunches')]

You can search the final directory in a path for items that match a search string using two functions: `.glob()` searches a single directory and `.rglob()` searches recursively. Both of these functions return generators. Similar to globbing in the Unix/Linux shell, only two meta-characters are recognized: `*` for one or more of any character and `?` for exactly one of any character.

In [18]:
# search the final directory in a path
shell_files = bio724.rglob("*.sh")
shell_files

<generator object Path.rglob at 0x31421a790>

In [17]:
# query the generator object
for f in shell_files:
    print(f)

/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/ecoli_feature_counts.sh
/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/template_feature_counts.sh
/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/yeast_feature_counts.sh
/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/multispecies_count.sh
/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/feature_counts.sh


In [13]:
# or, using a comprehension
[print(f) for f in shell_files]

/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/ecoli_feature_counts.sh
/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/template_feature_counts.sh
/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/yeast_feature_counts.sh
/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/multispecies_count.sh
/Users/gwray/Documents/Courses/Bio_724D_25/Lessons/18_Unix_scripting/working_scripts/feature_counts.sh


[None, None, None, None, None]